# 05. Tuning e Modelo Final

## 1. Importando bibliotecas

In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from src import data_prep, modeling, evaluation, interpretability, visualization

visualization.set_style()
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

RANDOM_STATE = 42
PROCESSED_DIR = ROOT / "data" / "processed"
MODELS_DIR = ROOT / "models"

In [2]:
from sklearn.model_selection import KFold, RandomizedSearchCV

X_train, X_test, y_train, y_test = data_prep.load_processed_split()
models = modeling.get_models()
cv_df = pd.read_csv(PROCESSED_DIR / "cv_results.csv")
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_df

,modelo,MAE_cv_media,MAE_cv_std,RMSE_cv_media,RMSE_cv_std,R2_cv_media,R2_cv_std
0,Ridge,286.588981,10.887599,363.730245,16.039247,0.577880,0.016967
1,Linear,286.594313,10.878159,363.732162,16.026899,0.577874,0.016981
2,GradientBoosting,298.356544,12.649975,376.590910,16.541578,0.547605,0.014073
3,RandomForest,306.669730,11.936137,387.894511,16.704744,0.519872,0.018383
4,XGBoost,333.548825,12.204408,420.430046,17.797170,0.435490,0.030693
5,DecisionTree,369.679236,15.868106,470.206177,22.543112,0.295061,0.019937


In [ ]:
def tune(name):

    dist = modeling.PARAM_DISTRIBUTIONS[name]
    if not dist:
        pipe = models[name]
        pipe.fit(X_train, y_train)
        return pipe, {}, float(cv_df.set_index("modelo").loc[name, "RMSE_cv_media"])
    search = RandomizedSearchCV(
        models[name], dist, n_iter=30, cv=kf, scoring="neg_root_mean_squared_error",
        random_state=RANDOM_STATE, n_jobs=-1,
    )
    search.fit(X_train, y_train)
    return search.best_estimator_, search.best_params_, float(-search.best_score_)

linear_names = [n for n in cv_df["modelo"] if n in ("Linear", "Ridge")]
tree_names = [n for n in cv_df["modelo"] if n not in ("Linear", "Ridge")]
best_linear_name = cv_df[cv_df["modelo"].isin(linear_names)].sort_values("RMSE_cv_media").iloc[0]["modelo"]
best_tree_name = cv_df[cv_df["modelo"].isin(tree_names)].sort_values("RMSE_cv_media").iloc[0]["modelo"]

tuned_linear, params_linear, cvrmse_linear = tune(best_linear_name)
tuned_tree, params_tree, cvrmse_tree = tune(best_tree_name)

print(f"Melhor linear: {best_linear_name} | params: {params_linear} | RMSE médio CV: {cvrmse_linear:.1f}")
print(f"Melhor árvore/boosting: {best_tree_name} | params: {params_tree} | RMSE médio CV: {cvrmse_tree:.1f}")

Melhor linear: Ridge | params: {'model__alpha': 10} | RMSE médio CV: 363.7
Melhor árvore/boosting: GradientBoosting | params: {'model__subsample': 1.0, 'model__n_estimators': 300, 'model__max_depth': 2, 'model__learning_rate': 0.05} | RMSE médio CV: 370.7


## 2. Seleção do modelo final

Foi comparado `cvrmse_linear` e `cvrmse_tree`, calculado acima, e escolhido o menor. O teste só entra depois, para reportar a métrica do modelo já escolhido e para o teste de significância a seguir.

In [4]:
candidates = {f"{best_linear_name} (tuned)": tuned_linear, f"{best_tree_name} (tuned)": tuned_tree}
candidates_cv_rmse = {f"{best_linear_name} (tuned)": cvrmse_linear, f"{best_tree_name} (tuned)": cvrmse_tree}

final_model_name = min(candidates_cv_rmse, key=candidates_cv_rmse.get)
final_model = candidates[final_model_name]

tuned_rows, tuned_preds = [], {}
for name, model in candidates.items():
    pred = model.predict(X_test)
    tuned_preds[name] = pred
    tuned_rows.append({"modelo": name, "RMSE_cv_treino": candidates_cv_rmse[name], **evaluation.regression_metrics(y_test, pred)})

tuned_df = pd.DataFrame(tuned_rows).sort_values("RMSE_cv_treino").reset_index(drop=True)
print("Comparação pós-tuning (RMSE de CV decide; teste só reporta):")
display(tuned_df)
print(f"\nModelo final escolhido (menor RMSE médio de CV): {final_model_name}")

Comparação pós-tuning (RMSE de CV decide; teste só reporta):


,modelo,RMSE_cv_treino,MAE,RMSE,R2,MAPE
0,Ridge (tuned),363.726954,288.256736,357.395979,0.527596,0.130926
1,GradientBoosting (tuned),370.664855,277.452411,346.564323,0.555797,0.127787



Modelo final escolhido (menor RMSE médio de CV): Ridge (tuned)


## 3. Teste de significância entre os dois candidatos tunados

Como a diferença de RMSE entre os modelos é pequena, foi usado bootstrap pareado com 5.000 reamostragens para verificar se essa diferença é realmente significativa ou apenas resultado do acaso.

In [5]:
name_a, name_b = list(candidates.keys())
boot = evaluation.bootstrap_compare_rmse(y_test, tuned_preds[name_a], tuned_preds[name_b])

print(f"{name_a}: RMSE teste = {boot['rmse_a']:.2f}")
print(f"{name_b}: RMSE teste = {boot['rmse_b']:.2f}")
print(f"Diferença observada ({name_a} - {name_b}): {boot['diff_observado']:.2f}")
print(f"IC 95% (bootstrap) da diferença: [{boot['ic95_diff'][0]:.2f}, {boot['ic95_diff'][1]:.2f}]")
print(f"p-valor aproximado: {boot['p_valor_aprox']:.3f}")
print("Diferença estatisticamente significativa a 95%?" , "Sim" if boot["diferenca_significativa_95"] else "Não")

Ridge (tuned): RMSE teste = 357.40
GradientBoosting (tuned): RMSE teste = 346.56
Diferença observada (Ridge (tuned) - GradientBoosting (tuned)): 10.83
IC 95% (bootstrap) da diferença: [-3.08, 24.47]
p-valor aproximado: 0.125
Diferença estatisticamente significativa a 95%? Não


- Não houve diferença estatisticamente significativa entre os dois modelos.
- Na prática, ambos apresentam desempenho preditivo equivalente.
- A escolha final considera:

    a. RMSE médio da validação cruzada;

    b. Interpretabilidade;
    
    c. Facilidade de manutenção.

## Saída desta etapa

In [6]:
import joblib

MODELS_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(final_model, MODELS_DIR / "modelo_final.joblib")

with open(PROCESSED_DIR / "modelo_final_nome.txt", "w") as f:
    f.write(final_model_name)

print("Modelo final salvo em:", MODELS_DIR / "modelo_final.joblib")

Modelo final salvo em: d:\mary-\Downloads\squad ds\squad-data-science\Case 3\models\modelo_final.joblib
